# event_log 변환 SQL 생성 (유저별 이벤트 전용)

`user_event_logs.sql` (generate_user_event_logs.ipynb 실행 결과)을 읽어,
`event_log` 테이블에 맞는 INSERT문으로 변환합니다.

## 매핑 대상 이벤트
- ad_click / ad_exposure → product 정보 + ad_id
- purchase_button_click → product 정보 + approved_amount
- coupon_received → coupon_code, discount_amount, expiry_date
- coupon_used → coupon_code, discount_amount

## 사용 방법
1. `user_event_logs.sql`을 이 노트북과 같은 디렉토리에 둔다
2. 실행하면 `event_log_insert_user_events.sql`이 생성된다

In [1]:
import re
import json
import os

In [2]:
SOURCE_SQL_FILE = 'user_event_logs.sql'
OUTPUT_SQL_FILE = 'event_log_insert_user_events.sql'

In [3]:
EVENT_LOG_COLUMNS = [
    'user_id', 'event_name', 'event_timestamp', 'product_id', 'product_name',
    'product_category', 'approved_amount', 'action_type', 'coupon_code',
    'discount_amount', 'expiry_date', 'search_keyword', 'page_name',
    'dwell_time', 'review_rating', 'earned_points', 'earn_reason',
    'login_id', 'created_at', 'ad_id', 'client_uuid'
]

In [4]:
def extract_rows_from_sql(sql_text):
    pattern = re.compile(
        r"\(\s*'([^']*(?:''[^']*)*)'\s*,\s*'((?:[^']|'')*)'\s*\)",
        re.DOTALL
    )
    rows = []
    for m in pattern.finditer(sql_text):
        history_ts = m.group(1).replace("''", "'")
        json_log   = m.group(2).replace("''", "'")
        rows.append((history_ts, json_log))
    return rows

def sql_literal(value):
    if value is None:
        return 'NULL'
    if isinstance(value, bool):
        return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)):
        return str(value)
    return f"'{str(value).replace(chr(39), chr(39)*2)}'"

def map_to_event_log(history_ts, json_log):
    data = json.loads(json_log)
    event_name = data.get('event_name')

    row = {col: None for col in EVENT_LOG_COLUMNS}

    # 공통
    row['user_id']         = data.get('user_id')
    row['event_name']      = event_name
    row['event_timestamp'] = data.get('event_timestamp')
    row['login_id']        = data.get('user_login_id')
    row['created_at']      = history_ts
    row['client_uuid']     = data.get('client_uuid')

    if event_name in ('ad_click', 'ad_exposure'):
        row['product_id']       = data.get('productId')
        row['product_name']     = data.get('productName')
        row['product_category'] = data.get('productCategory')
        row['ad_id']            = data.get('adId')

    elif event_name == 'purchase_button_click':
        row['product_id']       = data.get('productId')
        row['product_name']     = data.get('productName')
        row['product_category'] = data.get('productCategory')
        row['approved_amount']  = data.get('approvedAmount')

    elif event_name == 'coupon_received':
        row['coupon_code']     = data.get('couponCode')
        row['discount_amount'] = data.get('discountAmount')
        row['expiry_date']     = data.get('expiryDate')

    elif event_name == 'coupon_used':
        row['coupon_code']     = data.get('couponCode')
        row['discount_amount'] = data.get('discountAmount')

    return row

In [5]:
if not os.path.exists(SOURCE_SQL_FILE):
    raise FileNotFoundError(f'{SOURCE_SQL_FILE} 이 없습니다. generate_user_event_logs.ipynb를 먼저 실행해 주세요.')

with open(SOURCE_SQL_FILE, 'r', encoding='utf-8') as f:
    sql_text = f.read()

raw_rows       = extract_rows_from_sql(sql_text)
all_event_rows = [map_to_event_log(history_ts, json_log) for history_ts, json_log in raw_rows]

print(f'✅ {SOURCE_SQL_FILE} → {len(all_event_rows)}건 변환')

✅ user_event_logs.sql → 1417건 변환


In [6]:
columns_str = ', '.join(EVENT_LOG_COLUMNS)
lines  = [f'INSERT INTO event_log ({columns_str}) VALUES']
values = []

for row in all_event_rows:
    literals = [sql_literal(row[col]) for col in EVENT_LOG_COLUMNS]
    values.append('  (' + ', '.join(literals) + ')')

lines.append(',\n'.join(values) + ';')
event_log_sql = '\n'.join(lines)

with open(OUTPUT_SQL_FILE, 'w', encoding='utf-8') as f:
    f.write(event_log_sql)

print(f'✅ {len(all_event_rows)}건 event_log INSERT SQL 생성 완료 → {OUTPUT_SQL_FILE}')

✅ 1417건 event_log INSERT SQL 생성 완료 → event_log_insert_user_events.sql


In [7]:
print('=== EVENT_LOG INSERT SQL (앞 1000자) ===')
print(event_log_sql[:1000])

=== EVENT_LOG INSERT SQL (앞 1000자) ===
INSERT INTO event_log (user_id, event_name, event_timestamp, product_id, product_name, product_category, approved_amount, action_type, coupon_code, discount_amount, expiry_date, search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason, login_id, created_at, ad_id, client_uuid) VALUES
  (15, 'ad_click', '2026-06-04T15:14:43.000+09:00', '1677', '글로벌이노스 나이팅핏 TPE 요가매트', '스포츠/레저', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0001', '2026-06-04 15:14:44.000000', 43, '07ce894d-a2e2-43a6-be8f-7e2d1ab5c306'),
  (15, 'ad_click', '2026-06-16T17:00:48.000+09:00', '2012', '나이키 에어맥스 키높이 굽높은 어글리 여성 운동화 런닝화 HF3053 블랙', '패션잡화', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0001', '2026-06-16 17:00:49.000000', 78, '8608619d-c378-4d9a-b05f-5bdd48771895'),
  (15, 'ad_click', '2026-05-20T01:00:10.000+09:00', '2241', '토리든 다이브인 저분자 히알루론산 크림 80ml, 2개', '화장품/미용', NULL, NULL, NULL, NULL, NULL, NULL,